# GPT-OSS via LangChain + LangGraph

Ноутбук для проверки общих знаний `gpt-oss:120b` через `Ollama` на сервере из `.env`.

Здесь нет ручных HTTP-вызовов: запросы идут через `LangChain`, а запуск оформлен как маленький `LangGraph`.

Примечание: в этом окружении новый пакет `langchain_ollama` у меня возвращал `503`, поэтому здесь используется совместимая интеграция `langchain_community.chat_models.ChatOllama`, которая с этим сервером работает.

In [1]:
import os
import warnings
from pathlib import Path
from typing import TypedDict

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatOllama
from langgraph.graph import END, START, StateGraph

warnings.filterwarnings("ignore", message=".*ChatOllama.*deprecated.*")

d:\Programming\GitHub\nirma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = Path.cwd()
if not (repo_root / ".env").exists() and (repo_root.parent / ".env").exists():
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

BASE_URL = os.getenv("BASE_URL")
MODEL = os.getenv("CHAT_MODEL", "gpt-oss:120b")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

assert BASE_URL, "В .env не найден BASE_URL"

print({
    "repo_root": str(repo_root),
    "base_url": BASE_URL,
    "model": MODEL,
    "temperature": TEMPERATURE,
})

{'repo_root': 'd:\\Programming\\GitHub\\nirma', 'base_url': 'http://10.32.2.3:11434', 'model': 'gpt-oss:120b', 'temperature': 0.0}


In [3]:
llm = ChatOllama(
    model=MODEL,
    base_url=BASE_URL,
    temperature=TEMPERATURE,
)

system_prompt = """
Ты даешь взвешенные ответы по городской политике и стратегическому развитию.
Не выдумывай неизвестные факты.
Если не уверен в конкретных документах или не знаешь точных формулировок стратегии,
скажи об этом явно и опирайся на общие знания.
""".strip()

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

analysis_chain = analysis_prompt | llm | StrOutputParser()

In [4]:
class KnowledgeCheckState(TypedDict):
    question: str
    answer: str


def answer_question_node(state: KnowledgeCheckState):
    answer = analysis_chain.invoke({"question": state["question"]})
    return {"answer": answer}


graph = StateGraph(KnowledgeCheckState)
graph.add_node("answer_question", answer_question_node)
graph.add_edge(START, "answer_question")
graph.add_edge("answer_question", END)

app = graph.compile()

In [5]:
question = """
Как развивать город Гатчина с учетом ее стратегии социально-экономического развития?

Ответь как консультант по городскому развитию.
Сначала перечисли 5 ключевых направлений развития,
потом предложи 3 шага на горизонте 1-2 лет,
и 3 шага на горизонте 5-10 лет.

Если ты не знаешь точные положения официальной стратегии Гатчины,
скажи об этом явно и опирайся на общие знания о развитии городов,
о роли Гатчины в агломерации Санкт-Петербурга и о типичных целях
социально-экономического развития городов такого масштаба.
""".strip()

result = app.invoke({"question": question, "answer": ""})
display(Markdown(result["answer"]))

**Внимание:** У меня нет доступа к полному тексту официальной «Стратегии социально‑экономического развития города Гатчины», поэтому ниже я опираюсь на общедоступные сведения о городе, его месте в Санкт‑Петербургской агломерации и типичные приоритеты развития муниципальных образований подобного масштаба (население ≈ 150 тыс. человек, исторический центр, промышленный и административный потенциал, близость к крупному мегаполису). Если в официальных документах указаны иные формулировки, их следует уточнить и скорректировать предложенные меры.

---

## 5 ключевых направлений развития Гатчины

| № | Направление | Почему оно критично для Гатчины |
|---|-------------|--------------------------------|
| 1 | **Транспортная и логистическая интеграция** | Близость к Санкт‑Петербургу (≈ 45 км) делает город важным узлом пригородных и грузовых потоков. Улучшение связей повышает привлекательность для инвесторов и облегчает доступ к рабочим местам. |
| 2 | **Развитие инновационного и креативного кластера** | Наличие исторических объектов, университетов (например, филиалы СПбГУ, СПбГЭТУ) и научных центров (НИИ «Энергетика», «Теплоэнергетика») позволяет сформировать технологический парк, ориентированный на IT, биотехнологии, энергоэффективность. |
| 3 | **Туризм и культурно‑историческое наследие** | Гатчина – «королевская резиденция», в городе находятся дворец, парки, музеи. Их потенциал пока не полностью реализован; развитие туризма создаёт новые рабочие места и повышает доходы бюджета. |
| 4 | **Жилищно‑коммунальное развитие и качество городской среды** | Рост населения (в том числе за счёт переезда из СПб) требует обновления жилого фонда, повышения энергоэффективности зданий, развития зелёных зон и инфраструктуры для пешеходов/велосипедистов. |
| 5 | **Социальная инфраструктура и человеческий капитал** | Качество образования, медицины, спорта и культуры напрямую влияет на удержание и привлечение квалифицированных специалистов. Инвестиции в школы, поликлиники, детские сады и центры профессионального обучения – фундамент долгосрочного роста. |

---

## Шаги на горизонте **1‑2 года** (краткосрочные)

| № | Действие | Ожидаемый результат | Ключевые исполнители / партнёры |
|---|----------|----------------------|--------------------------------|
| 1 | **Запуск проекта «Умный пригородный транспорт»** – пилотные маршруты электробусов/троллейбусов, интеграция с СПб‑транспортом (единый билет, приложение). | Сокращение времени в пути до СПб, снижение выбросов, повышение комфорта для жителей и гостей. | Гатчинская администрация, Транспортный департамент СПб, частные операторы, инвесторы в электромобили. |
| 2 | **Создание «Технопарка Гатчина»** – подготовка площадки (реконверсия бывшего промышленного комплекса), стартовый набор грантов для стартапов в IT и энергоэффективных технологиях. | Привлечение 5‑10 молодых компаний, формирование первой «потоки» квалифицированных рабочих. | Администрация, Фонд «Развитие инноваций», местные ВУЗы, бизнес‑инкубаторы. |
| 3 | **Реставрация и открытие главных туристических объектов** (Дворец, парк, музей «Гатчинская резиденция») с современными сервисами (мультимедийные гайды, онлайн‑бронирование). | Увеличение туристического потока на 15‑20 % в сезон, рост доходов от билетов и сопутствующего сервиса. | Министерство культуры РФ, региональный департамент туризма, частные управляющие компании. |

---

## Шаги на горизонте **5‑10 лет** (долгосрочные)

| № | Действие | Ожидаемый результат | Ключевые исполнители / партнёры |
|---|----------|----------------------|--------------------------------|
| 1 | **Строительство кольцевой скоростной магистрали** (автодорога/трасса скоростного трамвая) соединяющей Гатчину с другими пригородными центрами (Павловск, Кронштадт) и СПб. | Увеличение пропускной способности, снижение нагрузки на центральные магистрали, привлечение новых логистических компаний. | Министерство транспорта РФ, региональные дорожные компании, инвесторы‑строители. |
| 2 | **Развитие «Экологически чистого жилого района»** – строительство 10 000 новых квартир с пассивным энергопотреблением, интеграция систем «умный дом», создание сети зарядных станций для электромобилей, развитие велосипедных дорожек. | Привлечение новых жителей (особенно молодых семей), снижение коммунальных расходов, повышение экологической репутации города. | Жилищные компании, энергетические компании, муниципальные службы ЖКХ, экологические НКО. |
| 3 | **Создание мультидисциплинарного образовательного кластера** – объединение вузов, техникумов и исследовательских центров в едином кампусе, запуск программ двойных дипломов с СПбГУ, СПбГЭТУ, а также международных партнёров. | Формирование квалифицированного кадрового резерва, рост числа выпускников, готовых работать в локальных инновационных компаниях. | ВУЗы, Министерство науки и высшего образования РФ, частные инвесторы в образование. |

---

### Как использовать предложенный план

1. **Провести аудит текущего состояния** – собрать данные по транспортным потокам, состоянию инфраструктуры, наличию свободных земельных участков, уровню занятости. Это позволит уточнить масштабы инвестиций и сроки реализации.
2. **Сформировать рабочие группы** по каждому направлению, включив представителей администрации, бизнеса, научных учреждений и гражданского общества. Регулярные встречи помогут корректировать план в реальном времени.
3. **Привлечь внешнее финансирование** – гранты федеральных программ «Развитие регионов», инвестиции частных фондов, публично‑частное партнёрство (PPP). Для крупных инфраструктурных проектов (трасса, технопарк) целесообразно использовать модели концессий.
4. **Оценивать эффективность** – установить KPI (например, сокращение среднего времени в пути, количество новых компаний в технопарке, рост туристических посещений, уровень энергоэффективности новых домов) и проводить ежегодный мониторинг.

---

**Итого:** предложенный набор приоритетов и конкретных шагов сочетает в себе улучшение транспортной доступности, развитие инновационной экономики, использование историко‑культурного потенциала, повышение качества городской среды и инвестирование в человеческий капитал. При их последовательной реализации Гатчина сможет укрепить свою роль в Санкт‑Петербургской агломерации, привлечь новые инвестиции и обеспечить устойчивый рост благосостояния жителей.

In [6]:
test_prompts = [
    "Какие сильные и слабые стороны у Гатчины как города-спутника Санкт-Петербурга?",
    "Какие меры обычно помогают среднему российскому городу удерживать молодежь?",
    "Предложи набор KPI для оценки качества городской среды и социальной инфраструктуры в Гатчине.",
]

for i, item in enumerate(test_prompts, start=1):
    print(f"\n=== Вопрос {i} ===\n{item}\n")
    answer = app.invoke({"question": item, "answer": ""})["answer"]
    print(answer)



=== Вопрос 1 ===
Какие сильные и слабые стороны у Гатчины как города-спутника Санкт-Петербурга?

**Гатчина — город‑спутник Санкт‑Петербурга.** Ниже — обобщённый обзор его сильных и слабых сторон, основанный на общедоступных данных (география, демография, инфраструктура, экономика, историко‑культурное наследие). Точных цифр из последних муниципальных стратегий у меня нет, поэтому в некоторых пунктах указаны лишь тенденции, а не конкретные показатели.

---

## Сильные стороны

| Сфера | Что работает в пользу Гатчины |
|-------|------------------------------|
| **Географическое положение** | • 45 км к юго‑западу от центра Санкт‑Петербурга – удобный доступ к мегаполису, но при этом город сохраняет более «пригородный» характер.<br>• Транспортные артерии (М-10 «Кольцевая», железная дорога «Ленинград‑Москва», автотрасса «Кольцевая»). |
| **Транспортная связь** | • Пригородные электрички (линия 2 «Ленинградская») ходят с интервалом 20–30 минут в часы пик, позволяя жителям работать в СПб и жит